# plugins

> A small plugin protocol for the top bar. Each plugin supplies an icon + name, an optional always-on background task, and a `render()` that builds the popup shown when its icon is activated (hover). Deliberately a **leaf module** -- it imports only external libs, never `cells.py`, so `cells.py` can `from .plugins import *` without a circular import (same one-directional rule as `llms.py`).

In [ ]:
#| default_exp plugins

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import threading, time, subprocess, sys, re, socket
from collections import deque
from fastcore.utils import *
from fasthtml.common import *
import fasthtml.components as fh
import psutil

## The plugin protocol

`Plugin` is the base class: subclass it, set `name`/`icon`/`trigger`, and implement `render()` for the hover popup (plus an optional always-on background task). `register` is a class decorator that adds an instance to `PLUGINS`; `start_plugins` kicks off each one's background task at server startup.

In [ ]:
#| export
class Plugin:
    "Base class for a top-bar plugin. Subclass it, set `name`/`icon`/`trigger` (or in __init__), and override `on_start`/`render` as needed; decorate the subclass with @register to add it to PLUGINS. `icon` is ICONS-style path data (plain 'd' strings and/or (d, scale) pairs) so cells.py renders it through the exact same SVG machinery as its built-in icons. `trigger` is 'hover' (the only kind wired up so far) or 'click'."
    name    = ''          # tooltip / panel-header text
    icon    = ()          # ICONS-style path tuple -- see _svg_icon() in cells.py
    trigger = 'hover'     # 'hover' | 'click'
    def on_start(self) -> None:
        "Optional lifecycle hook, called once at server startup (see start_plugins()). Long-running background work -- e.g. a sampling thread -- belongs here, NOT in __init__, so it only runs on a real server boot and never during import/tests."
        pass
    def render(self) -> FT:
        "Build the popup shown when this plugin is activated. Called at page-build time; reads whatever state on_start()'s background task has accumulated."
        return Div()

PLUGINS:list[Plugin] = []  # every registered plugin, in registration order; top_bar() renders them reversed (first-registered nearest the Tools icon)

def register(cls):
    "Class decorator: instantiate `cls` and append it to PLUGINS. Registration = importing this module. Returns `cls` unchanged."
    PLUGINS.append(cls())
    return cls

def start_plugins() -> None:
    "Call every registered plugin's on_start() once. Wired into cells.py's @app.on_event('startup') so background samplers spawn on real boot only."
    for p in PLUGINS: p.on_start()

## The system-monitor plugin

The first concrete plugin: `SystemMonitor` samples CPU/RAM/GPU/VRAM into ring buffers on a background thread and renders them as `_sparkline` mini-charts in its popup. GPU comes from `_gpu_sample` shelling out to `nvidia-smi` (the same approach as `slmn.misc.gpu_free`) on non-Mac hosts, or `_gpu_sample_darwin` reading Apple Silicon's IOKit registry on macOS.

In [ ]:
#| export
def _sparkline(vals:list, cls:str='text-cyan-400') -> FT:
    "A tiny line chart of `vals` (each 0-100): an SVG polyline (unchanged/untouched by the reference-box tweaks below -- same viewBox math throughout) layered over an independent CSS-bordered div for the 0-100% reference box. They're deliberately two separate elements, not one shared SVG viewBox, because the box needed an exact-pixel inset (2px each side) from the line's own span -- a fixed pixel amount can't be expressed reliably as a viewBox-unit margin shared with the line, since preserveAspectRatio=none's scale factor depends on the container's actual rendered width. The box's inset-1 (not inset-0) leaves 1px extra vertical breathing room for the line's stroke/round-caps top and bottom."
    W, H, MARGIN = 100, 30, 1  # small viewBox margin -- keeps the line's own stroke/round-caps off the *svg's* edges, independent of the separate box div below
    outer = 'relative w-full h-8'
    box_div = 'absolute inset-y-[1px] left-[2px] right-[2px] rounded-sm border border-base-content/40'
    viewbox = f'{-MARGIN} {-MARGIN} {W + 2*MARGIN} {H + 2*MARGIN}'
    if not vals: return Div(Div(cls=box_div), fh.Svg(viewbox=viewbox, cls=f'absolute inset-0 w-full h-full {cls}'), cls=outer)
    n = len(vals)
    def pt(i, v):
        x = 0 if n == 1 else i/(n-1)*W
        return f'{x:.2f},{H - max(0,min(100,v))/100*H:.2f}'
    pts = ' '.join(pt(i, v) for i, v in enumerate(vals)) if n > 1 else f'0,{H - vals[0]/100*H:.2f} {W},{H - vals[0]/100*H:.2f}'
    line = fh.Polyline(points=pts, fill='none', stroke='currentColor', stroke_width='1.5',
                       stroke_linejoin='round', stroke_linecap='round', vector_effect='non-scaling-stroke')
    svg = fh.Svg(line, viewbox=viewbox, preserveaspectratio='none', cls=f'absolute inset-0 w-full h-full {cls}')
    return Div(Div(cls=box_div), svg, cls=outer)

In [ ]:
#| export
# The rounded-square + zigzag "system monitor" glyph, traced from Scott's draw.io mockup. Lives
# here (not in cells.py's ICONS) because it belongs to this plugin -- see Plugin.icon / _svg_icon().
_ACTIVITY_ICON = ('M6 3h12A3 3 0 0 1 21 6v12A3 3 0 0 1 18 21H6A3 3 0 0 1 3 18V6A3 3 0 0 1 6 3Z',
                  'M3 12 5 12 9 17.5 15 6.5 19 12 21 12')

def _gpu_sample_darwin() -> float|None:
    "GPU utilization% on Apple Silicon via the IOKit registry's AGXAccelerator device -- no sudo required (unlike powermetrics), same technique remsysmon uses for local Mac GPU sampling. Unified memory means there's no separate VRAM pool to report, so this returns utilization only; SystemMonitor's VRAM row simply stays empty on a Mac, which render() already handles by skipping any metric with zero samples."
    try:
        out = subprocess.run(['ioreg', '-r', '-d', '1', '-c', 'AGXAccelerator', '-w', '0'],
                             capture_output=True, text=True, timeout=2)
        m = re.search(r'"Device Utilization %"=(\d+)', out.stdout)
        return float(m.group(1)) if m else None
    except Exception:
        return None

def _gpu_sample() -> tuple[float,float]|None:
    "(gpu_util%, vram_used%) for GPU 0, via nvidia-smi -- same shell-out approach as slmn.misc.gpu_free(), no pynvml/nvitop dependency needed. None if nvidia-smi is missing/fails (no GPU, driver issue, etc.) -- multi-GPU machines only get GPU 0 for now."
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=2)
        util, used, total = (float(x) for x in out.stdout.splitlines()[0].split(','))
        return util, (used / total * 100 if total else 0.0)
    except Exception:
        return None

@register
class SystemMonitor(Plugin):
    "First plugin: continuously logs CPU%, system-RAM%, GPU-util%, and VRAM% into ring buffers in the background (whether or not its panel is open), and plots the accumulated history as inline-SVG sparklines when hovered. GPU comes from nvidia-smi (_gpu_sample) on non-Mac hosts or the IOKit registry (_gpu_sample_darwin) on macOS; VRAM is nvidia-only -- Apple Silicon's unified memory has no separate pool to report. Rows just go empty/'--' if unavailable, no separate placeholder needed."
    name = 'System monitor'
    icon = _ACTIVITY_ICON
    def __init__(self, maxlen:int=300, interval:float=2.0):
        "maxlen samples at `interval` seconds ~= history length (300 x 2s ~= 10 min). Only sets up state; the sampling thread starts in on_start()."
        self.interval = interval
        self.cpu = deque(maxlen=maxlen)
        self.ram = deque(maxlen=maxlen)
        self.gpu = deque(maxlen=maxlen)
        self.vram = deque(maxlen=maxlen)
        self._started = False

    def on_start(self) -> None:
        "Spawn the (single, idempotent) daemon sampling thread."
        if self._started: return
        self._started = True
        threading.Thread(target=self._sample_loop, daemon=True).start()

    def _sample_loop(self) -> None:
        psutil.cpu_percent()  # prime: the first interval-less call always returns 0.0
        while True:
            try:
                self.cpu.append(psutil.cpu_percent())
                self.ram.append(psutil.virtual_memory().percent)
            except Exception: pass
            if sys.platform == 'darwin':
                if (u := _gpu_sample_darwin()) is not None: self.gpu.append(u)
            elif (g := _gpu_sample()) is not None:
                self.gpu.append(g[0]); self.vram.append(g[1])
            time.sleep(self.interval)

    def _metric_row(self, label:str, vals:deque, color:str) -> FT:
        "`color` (a text-* Tailwind class) is shared by the label, the percentage, and the sparkline itself, so a metric reads as one consistent color top to bottom."
        latest = f'{vals[-1]:.0f}%' if vals else '--'
        head = Div(Span(label, cls=f'text-xs font-semibold {color}'),
                   Span(latest, cls=f'text-xs ml-auto tabular-nums {color}'), cls='flex items-center')
        return Div(head, _sparkline(list(vals), color), cls='flex flex-col')

    def render(self) -> FT:
        # GPU on top, then VRAM, then system RAM, then CPU on the bottom -- colors per Scott: GPU
        # cyan, VRAM amber, RAM fuchsia, CPU emerald. A metric with no samples yet (e.g. GPU/VRAM
        # on a machine with no nvidia-smi, like a Mac) is skipped entirely rather than shown empty
        # -- the remaining rows just shift up to fill the list, no gaps.
        metrics = [('GPU', self.gpu, 'text-cyan-400'), ('VRAM', self.vram, 'text-amber-400'),
                   ('RAM', self.ram, 'text-fuchsia-400'), ('CPU', self.cpu, 'text-emerald-400')]
        rows = [self._metric_row(label, vals, color) for label, vals, color in metrics if vals]
        header = Div(Span('System monitor', cls='font-bold text-sm'), Span(socket.gethostname(), cls='text-xs opacity-60 ml-auto'), cls='flex items-baseline mb-1')
        return Div(header, *rows, cls='flex flex-col gap-2 p-2 w-72')

In [ ]:
# quick sanity check (does NOT start the sampler thread -- that only happens via on_start())
assert any(isinstance(p, SystemMonitor) for p in PLUGINS), "SystemMonitor should auto-register"
sm = next(p for p in PLUGINS if isinstance(p, SystemMonitor))
sm.cpu.extend([12, 34, 56, 30]); sm.ram.extend([40, 42, 41])
sm.gpu.extend([0, 5, 80]); sm.vram.extend([20, 21, 55])
panel = sm.render()             # should not raise
spark = _sparkline([10, 90, 50])
assert 'polyline' in to_xml(spark).lower()
assert 'System monitor' in to_xml(panel)
print('registered plugins:', [p.name for p in PLUGINS])
print('_gpu_sample() ->', _gpu_sample())  # None if no nvidia-smi on this box; a (util%, vram%) tuple otherwise
print('_gpu_sample_darwin() ->', _gpu_sample_darwin())  # None off macOS or if ioreg has no AGXAccelerator; a bare util% otherwise